# Entraînement d'un réseau de neurones pour les prédateurs

**Objectif** : remplacer la règle écrite à la main des prédateurs (`predator_heuristic` dans `policies.py`) par un réseau de neurones qui apprend seul à se déplacer pour survivre, rester nourri et se reproduire. Deux façons de formuler cet objectif sont disponibles (partie 4).

Personne ne connaît la « bonne » décision à chaque pas de temps : on ne peut donc pas apprendre en imitant des exemples. On utilise l'**apprentissage par renforcement** : les prédateurs essaient des actions, reçoivent une **récompense**, et le réseau est ajusté pour rendre plus probables les actions qui ont rapporté plus que prévu, et moins probables celles qui ont rapporté moins.

L'algorithme utilisé est **PPO** (*Proximal Policy Optimization*), un algorithme standard, robuste, adapté aux actions continues (ici : un virage et une vitesse).

**Tous les prédateurs utilisent le même réseau**, y compris ceux qui naissent pendant la simulation : chaque prédateur est une source d'expérience de plus pour l'entraîner. Les proies gardent leur comportement actuel (`prey_heuristic`).

Les étapes :
1. L'environnement : le scénario oscillant trouvé par `study.py`
2. Les observations : ce que perçoit un prédateur
3. Le réseau : comment il décide
4. La récompense : ce qu'on cherche à maximiser (deux objectifs au choix)
5. La collecte d'expérience
6. Les avantages : quelles actions ont été meilleures que prévu
7. La mise à jour du réseau (PPO)
8. La boucle d'entraînement
9. L'évaluation face à la règle écrite à la main, avec des indicateurs de coopération
10. La vidéo des prédateurs entraînés
11. La comparaison de tous les réseaux sauvegardés

Les briques réutilisables (observations, réseau, politique) sont dans `predator_brain.py`, pour pouvoir utiliser le réseau entraîné ailleurs (par exemple dans `Lotka-Volterra.ipynb`). Ce notebook contient tout ce qui sert à l'entraîner.

In [ ]:
import glob
import os
import time

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

from world import World
from policies import predator_heuristic
from predator_brain import (
    PredatorBrain, NeuralPredatorPolicy, predator_observations, concat_observations, to_tensors,
    actions_to_world, save_brain, load_brain, K_PREYS, K_PREDATORS, K_CARCASSES,
)

# the network is trained on the GPU if there is one
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

## Réglages

Tous les réglages sont regroupés ici. Les plus importants :
- `REWARD` : l'objectif, `"energy_level"` (par défaut) ou `"fitness"` (voir la partie 4). Chaque objectif a son propre fichier de réseau (`MODEL_PATH`), pour pouvoir les comparer.
- `TRAIN_MINUTES` : durée de l'entraînement. Le réseau est sauvegardé régulièrement dans `MODEL_PATH`.
- `RESUME` : `True` pour reprendre l'entraînement à partir du réseau déjà sauvegardé, au lieu de repartir de zéro.
- `EVAL_EVERY` : tous les combien d'itérations le réseau est évalué. La meilleure version est gardée dans `BEST_MODEL_PATH` (partie 8).
- `ENTROPY_COEF` : le bonus d'exploration (partie 7).

In [ ]:
# --- scenario: the parameters found by study.py (the same as in Lotka-Volterra.ipynb) ---
SCENARIO = dict(
    speed_prey=1.0, speed_predator=0.95,
    perception_prey=9., perception_predator=33., distance_danger=4.3,
    max_energy_prey=150, max_energy_predator=150,
    reproduction_energy_needed_prey=150, reproduction_energy_needed_predator=128,
    reproduction_energy_cost_prey=43, reproduction_energy_cost_predator=88,
    carcass_duration=60, carcass_eating_rate=13, carcass_max_share=0.5,
    energy_gain_prey=0.62, energy_loss_predator=0.4,
    carrying_capacity_prey=380, max_predators=250,
)
WORLD_SIZE = 86
N_PREYS_INIT = 190
N_PREDATORS_INIT = 20    # 7 in the scenario: more predators at the start gives more experience to learn from

# --- reward (see part 4): two objectives to choose from ---
# "energy_level": the energy of the predator at each time step, and a bonus for each reproduction;
#                 a part of the reward is the mean of all the predators of the world
# "fitness":      mostly the reproductions (the success of a predator in the sense of evolution), the energy
#                 being a secondary objective; a part of the reward is the mean of the predators around
REWARD = "energy_level"
GROUP_SHARE = 0.3            # part of the reward that comes from the other predators
if REWARD == "energy_level":
    GAMMA = 0.99             # discount: a reward k steps later counts GAMMA**k (horizon of about 100 steps)
    REPRODUCTION_BONUS = SCENARIO["reproduction_energy_cost_predator"] / SCENARIO["max_energy_predator"]
else:
    GAMMA = 0.995            # horizon of about 200 steps
    REPRODUCTION_BONUS = 1.0
    ENERGY_WEIGHT = 1.0      # weight of the energy, a secondary objective that also guides the learning

# --- analysis of the behaviour ---
COOPERATION_RADIUS = 5.0     # another predator closer than this is "nearby" (see part 9)

# --- collection of experience ---
N_WORLDS = 8                 # worlds simulated in parallel
ROLLOUT_STEPS = 256          # time steps simulated in each world between two updates of the network
EPISODE_STEPS = 2000         # a world is reset after this number of time steps (or when a species dies out)

# --- PPO ---
GAE_LAMBDA = 0.95
CLIP = 0.2
EPOCHS = 4
MINIBATCH = 8192
LEARNING_RATE = 3e-4
VALUE_COEF = 0.5
ENTROPY_COEF = 0.002         # exploration bonus: keeps the actions from becoming too predictable (see part 7)
MAX_GRAD_NORM = 0.5

# --- training ---
TRAIN_MINUTES = 30
RESUME = False
MODEL_PATH = f"./models/predator_brain_{REWARD}.pt"        # last version of the network (to resume the training)
BEST_MODEL_PATH = MODEL_PATH.replace(".pt", "_best.pt")    # best version according to the regular evaluation
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

# --- regular evaluation during the training, to keep the best network (see part 8) ---
EVAL_EVERY = 25              # iterations between two evaluations
EVAL_RUNS, EVAL_STEPS, EVAL_SEED = 4, 1000, 5000   # always the same worlds, to compare the versions fairly

## 1. L'environnement

C'est le scénario trouvé par l'étude d'hyperparamètres (`study.py`), dans lequel les populations oscillent comme dans un modèle de Lotka-Volterra.

Une carcasse contient l'énergie de la proie au moment de sa mort, et un prédateur ne peut en prendre qu'une part (`carcass_max_share`, la moitié) : le reste est pour les autres prédateurs.

Deux différences pendant l'entraînement :
- **20 prédateurs au départ au lieu de 7**. Au début, le réseau ne sait rien faire : ses prédateurs meurent de faim. Plus il y a de prédateurs, plus il y a d'expérience pour apprendre.
- **Le monde est remis à zéro** quand une espèce disparaît ou après `EPISODE_STEPS` pas de temps. Chaque simulation du début à la remise à zéro s'appelle un **épisode**.

Plusieurs mondes (`N_WORLDS`) sont simulés en parallèle : les situations vécues sont plus variées, ce qui rend l'apprentissage plus stable.

In [ ]:
def new_world(n_predators=N_PREDATORS_INIT):
    # create a world with the scenario and its initial agents
    world = World(WORLD_SIZE, WORLD_SIZE)
    for name, value in SCENARIO.items():
        setattr(world, name, value)
    world.add_prey(number=N_PREYS_INIT)
    world.add_predator(number=n_predators)
    return world


world = new_world()
print(len(world.preys), "preys,", len(world.predators), "predators")

## 2. Les observations : ce que perçoit un prédateur

À chaque pas de temps, chaque prédateur reçoit les informations suivantes (calculées par `predator_observations` dans `predator_brain.py`) :

| Entrée | Contenu |
|---|---|
| **Lui-même** (8 valeurs) | position (sinus et cosinus de x et de y, car le monde est torique : le bord droit touche le bord gauche), orientation (sinus, cosinus), énergie, compte à rebours après une naissance |
| **Proies** (8 au plus) | position, distance, orientation |
| **Prédateurs** (6 au plus) | position, distance, orientation, **énergie** |
| **Carcasses** (4 au plus) | position, distance, énergie restante, **énergie que lui peut encore prendre** (un prédateur ne peut prendre que `carcass_max_share` de chaque carcasse) |

Les voisins sont **les plus proches dans le rayon de perception** (`perception_predator` = 33). Ils sont vus **depuis le prédateur** : « 10 devant et 3 à gauche » plutôt que des coordonnées absolues. Une même situation (une proie juste devant) donne ainsi la même entrée, où que le prédateur soit et quelle que soit son orientation, ce qui facilite beaucoup l'apprentissage. Les distances sont divisées par le rayon de perception pour rester entre -1 et 1.

Le nombre de voisins varie. Les cases vides sont remplies de 0 et signalées par un **masque** (`True` = il y a un voisin), pour que le réseau les ignore.

In [ ]:
obs = predator_observations(world)
for key, value in obs.items():
    print(f"{key:16s} {str(value.shape):16s} {value.dtype}")

# what the first predator perceives
i = 0
print("\npredator", i, ": energy", round(world.predators.energy[i]), "| preys perceived:", obs["preys_mask"][i].sum(),
      "| predators:", obs["predators_mask"][i].sum(), "| carcasses:", obs["carcasses_mask"][i].sum())

# drawing: the predator, its perception radius and the neighbours it perceives
fig, (ax, ax_local) = plt.subplots(1, 2, figsize=(12, 5.5))
p = world.predators.pos[i]
ax.scatter(world.preys.x, world.preys.y, s=8, color="blue", label="preys")
ax.scatter(world.predators.x, world.predators.y, s=25, color="orange", label="predators")
ax.scatter(*p, s=80, color="red", label="predator observed")
ax.add_patch(plt.Circle(p, world.perception_predator, fill=False, color="red", ls="--"))
ax.set_xlim(0, world.width); ax.set_ylim(0, world.height); ax.set_aspect("equal")
ax.set_title("In the world (perception radius dashed)"); ax.legend(loc="upper right", fontsize=8)

# the same neighbours, as the network sees them (in front of the predator = up)
for key, color in (("preys", "blue"), ("predators", "orange")):
    found = obs[key + "_mask"][i]
    front, left = obs[key][i, found, 0], obs[key][i, found, 1]
    ax_local.scatter(-left, front, color=color, label=f"{key} ({found.sum()})")
ax_local.scatter(0, 0, marker="^", s=150, color="red", label="predator")
ax_local.add_patch(plt.Circle((0, 0), 1, fill=False, color="red", ls="--"))
ax_local.set_xlim(-1.1, 1.1); ax_local.set_ylim(-1.1, 1.1); ax_local.set_aspect("equal")
ax_local.set_xlabel("right  (/ perception radius)"); ax_local.set_ylabel("in front  (/ perception radius)")
ax_local.set_title("Seen by the network (closest neighbours only)"); ax_local.legend(fontsize=8)
plt.show()

## 3. Le réseau : comment il décide

Le réseau (`PredatorBrain` dans `predator_brain.py`) se lit de gauche à droite :

```
lui-même ───────────► petit réseau ──────────────┐
proies (8 au plus) ─► même petit réseau pour      │
                      chaque proie, puis moyenne ├─► acteur  ─► moyenne du virage et de la vitesse
                      et maximum sur les proies  │
prédateurs (6) ─────► idem                       ├─► critique ─► valeur de la situation
carcasses (4) ──────► idem ──────────────────────┘
```

- **Les voisins sont traités comme des ensembles** (`SetEncoder`) : chaque voisin passe dans le même petit réseau, puis on fait la moyenne et le maximum sur les voisins présents. Le résultat ne dépend donc ni de l'ordre des voisins ni de leur nombre.
- **L'acteur** donne la décision. Pendant l'entraînement, l'action est tirée au hasard selon une loi normale centrée sur cette décision, dont l'écart-type est aussi appris. Ce hasard, c'est l'**exploration** : le prédateur essaie des variantes de ce qu'il pense être le mieux. Une fois entraîné, on prend directement la décision moyenne.
- **Le critique** estime la **valeur** de la situation : les récompenses que le prédateur peut encore espérer. Il ne sert que pendant l'entraînement, pour juger si une action a été meilleure ou moins bonne que prévu (partie 6).

**Les actions** : le réseau sort deux nombres, transformés par `actions_to_world` :
- le **virage**, entre -1 et 1 (une fraction du virage maximal de π/8 par pas) ;
- la **vitesse**, entre 0 (immobile) et 1 (`speed_predator`, la vitesse actuelle des prédateurs).

La perte d'énergie par pas de temps reste la même quelle que soit la vitesse.

In [ ]:
brain = PredatorBrain().to(device)
print(brain)
print("\nnumber of parameters:", sum(p.numel() for p in brain.parameters()))

# test on the observations of the world above
with torch.no_grad():
    distribution, value = brain(to_tensors(obs, device))
print("\nmean actions of the first predators (turn, speed before transformation):\n", distribution.mean[:3].cpu().numpy())
print("actions given to the world (turn in [-1, 1], speed in [0, 1]):\n", actions_to_world(distribution.mean[:3].cpu().numpy()))

## 4. La récompense : ce qu'on cherche à maximiser

Deux objectifs sont disponibles (`REWARD` dans les réglages). Ils ont chacun leur fichier de réseau, pour pouvoir les entraîner tous les deux et les comparer (partie 11).

### Objectif `"energy_level"` (par défaut) : énergie et reproductions

À chaque pas de temps, chaque prédateur reçoit :

$$r_\text{individuelle} = (1-\gamma)\,\frac{E}{E_\max} \;+\; B \times \text{(1 s'il s'est reproduit pendant ce pas, sinon 0)}, \qquad \gamma = 0{,}99$$

- **L'énergie** : $E$ est son énergie après le pas, divisée par l'énergie maximale. Avec le facteur $(1-\gamma)$, la somme des récompenses futures pondérées par $\gamma^k$ vaut l'énergie moyenne du prédateur dans le futur.
- **La reproduction** : le bonus $B$ = 88 / 150 = 0,59 vaut ce que rapporterait l'énergie dépensée pour se reproduire si le prédateur la gardait pour toujours.

La part collective est la moyenne de **tous les prédateurs du monde**.

**La limite de cet objectif** : il récompense surtout le fait de **rester bien nourri**. Avec les réseaux entraînés, environ les trois quarts de la récompense viennent de l'énergie. Un prédateur bien nourri qui se reproduit peu rapporte à peu près autant qu'un prédateur plus affamé qui se reproduit beaucoup : le comportement appris dépend donc fortement du réglage de $B$.

### Objectif `"fitness"` : surtout la descendance

En évolution, la réussite d'un individu se mesure à sa **descendance** : l'énergie n'est qu'un moyen de survivre et de se reproduire. Cet objectif fait donc de la reproduction l'objectif principal, et garde l'énergie comme objectif secondaire :

$$r_\text{individuelle} = B \times \text{(reproduction)} \;+\; w\,(1-\gamma)\,\frac{E}{E_\max}, \qquad B = 1,\quad w = 1,\quad \gamma = 0{,}995$$

- **La reproduction pèse plus** : une reproduction vaut autant que l'énergie maximale gardée pour toujours (contre 0,59 fois avec `"energy_level"`). Au niveau de la règle écrite à la main (environ 3,6 reproductions pour 1000 pas), la reproduction fait plus de la moitié de la récompense, au lieu d'un quart à un tiers.
- **L'énergie reste un vrai objectif, secondaire** : elle guide l'apprentissage, car le réseau gagne quelque chose dès qu'il apprend à manger, bien avant de réussir sa première reproduction.

**Résultat de la comparaison** (un entraînement de 18 minutes par objectif, environ 3 millions de transitions) : avec cet objectif, le réseau apprend **moins bien** qu'avec `"energy_level"`. Il se reproduit moins, et les prédateurs disparaissent. Pour se reproduire, il faut d'abord savoir chasser, et c'est l'énergie qui l'apprend au réseau. Ici, son poids par pas est deux fois plus faible ($1-\gamma$ = 0,005 au lieu de 0,01), et l'horizon plus long rend l'apprentissage plus bruité. Cet objectif reste disponible pour des entraînements plus longs, ou pour affiner un réseau déjà entraîné avec `"energy_level"`.

**Une première version, abandonnée** : l'énergie n'y était qu'un *façonnage par potentiel* (récompenser la variation d'énergie, $\gamma E_{t+1} - E_t$). En théorie, ce terme ne change pas le comportement optimal et aide seulement à apprendre. En pratique, le critique apprend vite à le prédire (l'énergie fait partie de ses entrées) et l'annule dans les avantages. Il ne restait alors que la récompense de reproduction, trop rare pour un réseau débutant, qui ne se reproduit jamais : en 18 minutes, le réseau n'avait rien appris.

- **Un horizon plus long** : $\gamma = 0{,}995$, soit environ 200 pas au lieu de 100. Regagner l'énergie d'une reproduction prend plusieurs centaines de pas.
- **Une part collective locale** : la moyenne porte sur les **prédateurs dans son rayon de perception** (lui compris), ceux avec qui il peut réellement coopérer. Avec 30 à 80 prédateurs dans le monde, la moyenne de tout le monde ne dépend presque pas de ce qu'il fait : pour l'apprentissage, c'est surtout du bruit.

### Dans les deux cas

$$r = (1-\alpha)\, r_\text{individuelle} + \alpha \times \text{moyenne des } r_\text{individuelle} \text{ du groupe}, \qquad \alpha = \text{GROUP\_SHARE} = 0{,}3$$

Un prédateur mort ne reçoit plus rien : mourir est naturellement pénalisé, puisqu'il perd tout ce qu'il aurait pu obtenir ensuite.

`step_and_reward` fait avancer un monde d'un pas et calcule la récompense de chaque prédateur. Elle sert à la fois pendant l'entraînement et pendant l'évaluation. `cooperation_events` relève ce que les prédateurs ont fait ensemble pendant le pas (partie 9).

In [ ]:
def compute_rewards(world, positions, energy_before, energy_after, reproduced):
    # rewards of the predators that were in the world at the beginning of the step
    # positions, energy_before: before the step; energy_after: after the step (0 if the predator died)
    # reproduced: number of reproductions during the step (0 or 1)
    # return: the (mixed) reward of each predator, and its individual part
    max_energy = world.max_energy_predator
    if REWARD == "energy_level":
        individual = (1 - GAMMA) * energy_after / max_energy + REPRODUCTION_BONUS * reproduced
        group = np.full(len(individual), individual.mean() if len(individual) else 0.0)
    else:
        individual = REPRODUCTION_BONUS * reproduced + ENERGY_WEIGHT * (1 - GAMMA) * energy_after / max_energy
        # mean over the predators around (within the perception radius, the predator itself included)
        dx, dy = world.displacement(positions, positions)
        around = np.hypot(dx, dy) <= world.perception_predator
        group = around @ individual / np.maximum(around.sum(axis=1), 1)
    return (1 - GROUP_SHARE) * individual + GROUP_SHARE * group, individual


def step_and_reward(world, predator_actions=None):
    # advance the world by one time step (predator_actions=None: world.predator_policy decides),
    # and compute the reward of each predator that was in the world at the beginning of the step
    predators = world.predators
    ids, positions = predators.ids.copy(), predators.pos.copy()
    energy_before, reproductions = predators.energy.copy(), predators.reproductions.copy()
    out = world.step(predator_actions=predator_actions)

    # what happened to each predator (the predators stay sorted by id, the dead ones are missing)
    position = np.searchsorted(world.predators.ids, ids)
    alive = position < len(world.predators)
    alive[alive] = world.predators.ids[position[alive]] == ids[alive]
    energy_after, reproduced = np.zeros(len(ids)), np.zeros(len(ids))
    energy_after[alive] = world.predators.energy[position[alive]]
    reproduced[alive] = world.predators.reproductions[position[alive]] - reproductions[alive]

    rewards, individual = compute_rewards(world, positions, energy_before, energy_after, reproduced)
    return dict(out=out, alive=alive, position=position, rewards=rewards, individual=individual,
                energy=energy_after, reproduced=reproduced)


def cooperation_events(world):
    # what the predators did together during the last time step (counts, see part 9)
    predators = world.predators
    n = len(predators)
    killer_ids = world.last_kills["killer_ids"]
    events = dict(predator_steps=n, grouped=0, kills=len(killer_ids), group_kills=0, meals=0, shared_meals=0)

    if n > 1:
        # predators that have another predator closer than COOPERATION_RADIUS
        dx, dy = world.displacement(predators.pos, predators.pos)
        near = np.hypot(dx, dy) < COOPERATION_RADIUS
        np.fill_diagonal(near, False)
        has_neighbour = near.any(axis=1)
        events["grouped"] = int(has_neighbour.sum())
        # the same for the predators that killed a prey during this step
        killers = np.searchsorted(predators.ids, killer_ids)
        found = killers < n
        found[found] = predators.ids[killers[found]] == killer_ids[found]
        events["group_kills"] = int(has_neighbour[killers[found]].sum())

    # predators eating a carcass with another predator
    meals = world.last_meals
    events["meals"] = len(meals)
    events["shared_meals"] = int((np.bincount(meals)[meals] >= 2).sum()) if len(meals) else 0
    return events


def cooperation_indicators(counts):
    # from the counts summed over many steps: share of the kills made with another predator nearby,
    # share of the time a predator has another one nearby, their ratio, and share of the meals shared
    group_kills = counts["group_kills"] / max(counts["kills"], 1)
    grouped = counts["grouped"] / max(counts["predator_steps"], 1)
    return dict(group_kills=group_kills, grouped=grouped,
                hunting_together=group_kills / grouped if grouped > 0 else np.nan,
                shared_meals=counts["shared_meals"] / max(counts["meals"], 1))


print("objective:", REWARD, "| GAMMA:", GAMMA, "| bonus for a reproduction:", round(REPRODUCTION_BONUS, 3))

## 5. La collecte d'expérience

On fait jouer le réseau pendant `ROLLOUT_STEPS` pas de temps dans chaque monde. À chaque pas :
1. chaque prédateur **observe** ;
2. le réseau **décide** pour tous les prédateurs de tous les mondes à la fois (un seul calcul sur le GPU) ;
3. chaque monde **avance** d'un pas, et chaque prédateur reçoit sa **récompense** (`step_and_reward`).

On garde chaque **transition** (observation, action, probabilité de l'action, valeur estimée, récompense) pour la mise à jour du réseau.

**Suivre chaque prédateur** : pour juger une action, il faut savoir ce qui est arrivé ensuite **au même prédateur**. Or des prédateurs meurent et naissent à chaque pas. Chaque prédateur a donc un identifiant unique (`world.predators.ids`), qui permet de relier chacune de ses transitions à la suivante (`next_index`). Trois cas peuvent se produire :
- **Le prédateur meurt** : sa trajectoire se termine (`done`). Il n'y a plus de récompense à attendre.
- **L'épisode ou la collecte s'arrête alors qu'il est vivant** : sa trajectoire est coupée. Les récompenses qu'il aurait reçues ensuite sont estimées par le critique (`bootstrap`).
- **Un prédateur naît** : il commence une nouvelle trajectoire au pas suivant.

In [ ]:
class TrainingWorld:
    # a world used for the training, and the link between each predator and its last transition

    def __init__(self):
        self.reset()

    def reset(self):
        self.world = new_world()
        self.steps = 0
        # index of the last transition of each predator (in the order of world.predators), -1 if none
        self.last = np.full(len(self.world.predators), -1)

    def finished(self):
        return len(self.world.preys) == 0 or len(self.world.predators) == 0 or self.steps >= EPISODE_STEPS


def cut_trajectories(training_world, brain, bootstrap):
    # the predators still alive stop their trajectory here: their future rewards are estimated by the critic
    has_transition = training_world.last >= 0
    if len(training_world.world.predators) == 0 or not has_transition.any():
        return
    with torch.no_grad():
        _, value = brain(to_tensors(predator_observations(training_world.world), device))
    bootstrap.append((training_world.last[has_transition], value.cpu().numpy()[has_transition]))
    training_world.last[:] = -1


def collect_rollout(training_worlds, brain, steps=ROLLOUT_STEPS):
    # simulate `steps` time steps in every world, the predators being controlled by the network
    # return: the transitions (one per predator and per time step) and statistics
    observations, raw_actions, log_probs, values, rewards = [], [], [], [], []
    links, deaths, bootstrap, step_ranges = [], [], [], []
    totals = dict(energy=0.0, individual_reward=0.0, reproductions=0, deaths=0, speed=0.0)
    cooperation = dict(predator_steps=0, grouped=0, kills=0, group_kills=0, meals=0, shared_meals=0)
    episode_lengths = []
    n = 0  # number of transitions so far

    for _ in range(steps):
        # 1-2. every predator of every world observes, and the network decides for all of them at once
        counts = [len(tw.world.predators) for tw in training_worlds]
        obs = concat_observations([predator_observations(tw.world) for tw in training_worlds])
        with torch.no_grad():
            distribution, value = brain(to_tensors(obs, device))
            raw = distribution.sample()
            log_prob = distribution.log_prob(raw).sum(-1)
        raw = raw.cpu().numpy()
        actions = actions_to_world(raw)

        observations.append(obs)
        raw_actions.append(raw)
        log_probs.append(log_prob.cpu().numpy())
        values.append(value.cpu().numpy())
        step_ranges.append((n, n + sum(counts)))
        step_rewards = np.zeros(sum(counts), dtype=np.float32)

        # 3. each world advances by one time step, and the predators get their reward
        offset = 0
        for tw, count in zip(training_worlds, counts):
            mine = slice(offset, offset + count)
            index = np.arange(n + offset, n + offset + count)  # indices of the transitions of this world
            offset += count

            # link each predator to its previous transition
            has_previous = tw.last >= 0
            links.append((tw.last[has_previous], index[has_previous]))

            result = step_and_reward(tw.world, actions[mine])
            tw.steps += 1
            alive = result["alive"]
            step_rewards[mine] = result["rewards"]
            deaths.append(index[~alive])

            # the survivors continue their trajectory, the newborns will start a new one
            tw.last = np.full(len(tw.world.predators), -1)
            tw.last[result["position"][alive]] = index[alive]

            totals["energy"] += result["energy"][alive].sum()
            totals["individual_reward"] += result["individual"].sum()
            totals["reproductions"] += int(result["reproduced"].sum())
            totals["deaths"] += int((~alive).sum())
            totals["speed"] += actions[mine, 1].sum()
            for key, event_count in cooperation_events(tw.world).items():
                cooperation[key] += event_count

            # end of the episode
            if tw.finished():
                cut_trajectories(tw, brain, bootstrap)
                episode_lengths.append(tw.steps)
                tw.reset()

        rewards.append(step_rewards)
        n += sum(counts)

    # end of the collection: the trajectories still running are cut too
    for tw in training_worlds:
        cut_trajectories(tw, brain, bootstrap)

    # put everything together
    next_index = np.full(n, -1)
    for previous, new in links:
        next_index[previous] = new
    done = np.zeros(n, dtype=bool)
    done[np.concatenate(deaths)] = True
    bootstrap_value = np.zeros(n, dtype=np.float32)
    for index, value in bootstrap:
        bootstrap_value[index] = value

    batch = dict(
        obs=concat_observations(observations), raw=np.concatenate(raw_actions),
        log_prob=np.concatenate(log_probs), value=np.concatenate(values), reward=np.concatenate(rewards),
        next_index=next_index, done=done, bootstrap=bootstrap_value, step_ranges=step_ranges,
    )
    alive_steps = n - totals["deaths"]
    stats = dict(
        transitions=n,
        predators_per_world=n / (steps * len(training_worlds)),
        energy=totals["energy"] / max(alive_steps, 1),
        individual_reward=totals["individual_reward"] / n,
        reproductions_per_1000=1000 * totals["reproductions"] / n,
        deaths_per_1000=1000 * totals["deaths"] / n,
        speed=totals["speed"] / n,
        **cooperation_indicators(cooperation),
        episode_length=np.mean(episode_lengths) if episode_lengths else np.nan,
    )
    return batch, stats


# test with a small collection
test_worlds = [TrainingWorld() for _ in range(2)]
test_batch, test_stats = collect_rollout(test_worlds, brain, steps=20)
print(test_batch["raw"].shape[0], "transitions |", {k: round(float(v), 3) for k, v in test_stats.items()})

## 6. Les avantages : quelles actions ont été meilleures que prévu ?

Pour chaque transition, on compare **ce que l'action a rapporté** à **ce que le critique prévoyait** :

$$\delta_t = r_t + \gamma\, V(s_{t+1}) - V(s_t)$$

- $r_t + \gamma V(s_{t+1})$ : la récompense reçue, plus la valeur de la situation où l'action a mené ;
- $V(s_t)$ : la valeur prévue avant d'agir.

Si $\delta_t > 0$, l'action a été meilleure que prévu.

L'**avantage** $A_t$ cumule ces écarts sur la suite de la trajectoire du même prédateur (méthode **GAE**, *Generalized Advantage Estimation*) :

$$A_t = \delta_t + \gamma\lambda\, A_{t+1}$$

Le paramètre $\lambda$ (`GAE_LAMBDA` = 0,95) règle le compromis entre une estimation à court terme (peu bruitée mais biaisée par les erreurs du critique) et une estimation à long terme (l'inverse).

- **Si le prédateur meurt**, $V(s_{t+1}) = 0$.
- **Si sa trajectoire a été coupée**, $V(s_{t+1})$ est la valeur estimée au moment de la coupure.

Le calcul se fait à rebours, du dernier pas au premier. L'avantage d'une transition dépend de celui de la transition suivante du même prédateur, que l'on retrouve grâce à `next_index`.

La cible du critique est $A_t + V(s_t)$ : ce que la transition a réellement rapporté, au sens de cette estimation.

In [ ]:
def compute_advantages(batch):
    value, reward, next_index = batch["value"], batch["reward"], batch["next_index"]
    has_next = next_index >= 0
    # value of the next situation: 0 if the predator died, the critic's estimate if the trajectory was cut
    next_value = np.where(has_next, value[next_index], np.where(batch["done"], 0.0, batch["bootstrap"]))
    delta = reward + GAMMA * next_value - value

    advantages = np.zeros_like(delta)
    for start, end in reversed(batch["step_ranges"]):  # backwards in time, one time step at a time
        following = next_index[start:end]
        advantages[start:end] = delta[start:end] + GAMMA * GAE_LAMBDA * np.where(
            following >= 0, advantages[following], 0.0
        )
    batch["advantages"] = advantages.astype(np.float32)
    batch["returns"] = (advantages + value).astype(np.float32)


compute_advantages(test_batch)
print("advantages: mean", test_batch["advantages"].mean().round(4), "std", test_batch["advantages"].std().round(4))

## 7. La mise à jour du réseau (PPO)

On ajuste le réseau sur les transitions collectées, en plusieurs passes (`EPOCHS`) sur des petits paquets (`MINIBATCH`).

**L'acteur** : pour chaque transition, on compare la probabilité de l'action avec le réseau actuel et avec le réseau qui l'a choisie : $\rho = p_\text{nouveau}(a) / p_\text{ancien}(a)$. On veut augmenter $\rho \times A$ :
- rendre plus probables les actions à avantage positif ;
- rendre moins probables celles à avantage négatif.

La particularité de PPO : $\rho$ est **bornée** entre $1 - 0{,}2$ et $1 + 0{,}2$ (`CLIP`). Une mise à jour ne peut donc pas trop changer le comportement d'un coup, ce qui rend l'apprentissage stable :

$$L_\text{acteur} = -\,\text{moyenne}\big(\min(\rho A,\; \text{clip}(\rho, 1-\epsilon, 1+\epsilon)\,A)\big)$$

**Le critique** : il apprend à prédire la cible calculée à l'étape 6, avec une erreur quadratique.

Les avantages sont **normalisés** (moyenne 0, écart-type 1) dans chaque paquet. L'échelle des récompenses ne change ainsi pas la vitesse d'apprentissage.

**Le bonus d'exploration** : la perte contient aussi $-\,$`ENTROPY_COEF` $\times$ l'entropie des actions, qui mesure leur part de hasard. Il récompense un peu le fait de garder de la variété dans les actions.
- **Sans lui**, l'écart-type des actions diminue sans cesse : de 0,5 à 0,09 en 2 heures d'entraînement. Le réseau n'essaie alors presque plus de variantes, et ne peut plus corriger ses défauts : lors de cet entraînement, il s'était dégradé après 30 millions de transitions.
- **La valeur 0,002 est petite** : elle freine cette diminution sans empêcher le réseau de devenir précis là où c'est utile.

On suit aussi deux indicateurs de stabilité :
- **`kl`** : à quel point le comportement a changé. Il doit rester petit, de l'ordre de 0,01.
- **`std`** : l'écart-type des actions, c'est-à-dire la quantité d'exploration.

In [ ]:
def ppo_update(brain, optimizer, batch):
    obs = to_tensors(batch["obs"], device)
    raw = torch.as_tensor(batch["raw"], device=device)
    old_log_prob = torch.as_tensor(batch["log_prob"], device=device)
    advantages = torch.as_tensor(batch["advantages"], device=device)
    returns = torch.as_tensor(batch["returns"], device=device)
    n = len(raw)
    logs = dict(policy_loss=[], value_loss=[], kl=[], clipped=[])

    for _ in range(EPOCHS):
        order = torch.randperm(n, device=device)
        for start in range(0, n, MINIBATCH):
            idx = order[start:start + MINIBATCH]
            distribution, value = brain({key: tensor[idx] for key, tensor in obs.items()})
            log_prob = distribution.log_prob(raw[idx]).sum(-1)
            ratio = (log_prob - old_log_prob[idx]).exp()

            adv = advantages[idx]
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)
            policy_loss = -torch.min(ratio * adv, ratio.clamp(1 - CLIP, 1 + CLIP) * adv).mean()
            value_loss = (value - returns[idx]).pow(2).mean()
            entropy = distribution.entropy().sum(-1).mean()
            loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(brain.parameters(), MAX_GRAD_NORM)
            optimizer.step()

            with torch.no_grad():
                logs["policy_loss"].append(policy_loss.item())
                logs["value_loss"].append(value_loss.item())
                logs["kl"].append((old_log_prob[idx] - log_prob).mean().item())
                logs["clipped"].append(((ratio - 1).abs() > CLIP).float().mean().item())

    logs = {key: float(np.mean(values)) for key, values in logs.items()}
    logs["std"] = brain.log_std.exp().mean().item()
    return logs

## 8. La boucle d'entraînement

On répète **collecte → avantages → mise à jour** jusqu'à la fin du temps prévu (`TRAIN_MINUTES`). Le réseau est sauvegardé régulièrement dans `MODEL_PATH`.

**La référence** : on mesure d'abord les mêmes indicateurs avec la règle écrite à la main (`predator_heuristic`), dans les mêmes conditions. Cela donne le niveau à battre sur les courbes.

Ce qu'on surveille :
- **`energy`** : énergie moyenne des prédateurs vivants ;
- **`repro/1000`** : reproductions pour 1000 pas de vie de prédateur ;
- **`deaths/1000`** : morts de faim pour 1000 pas de vie ;
- **`reward`** : récompense individuelle moyenne par pas. Elle n'est comparable qu'entre réseaux entraînés avec le même objectif ;
- **`together`** : l'indice de chasse groupée (voir la partie 9) ;
- **`episode`** : durée moyenne des épisodes. Elle augmente si les prédateurs survivent plus longtemps sans exterminer les proies.

Au début, les prédateurs se déplacent presque au hasard et meurent vite. Les épisodes sont courts.

**Garder le meilleur réseau** : un long entraînement ne s'améliore pas toujours jusqu'au bout. Lors d'un entraînement de 2 heures, le réseau avait culminé vers 20 à 35 millions de transitions, puis s'était lentement dégradé. Tous les `EVAL_EVERY` itérations, le réseau est donc évalué **sans exploration** (il prend sa décision moyenne), toujours sur les **mêmes mondes** (`EVAL_RUNS` mondes de `EVAL_STEPS` pas). Le score est la récompense individuelle moyenne par pas, c'est-à-dire l'objectif lui-même.
- **S'il bat le meilleur score**, le réseau est sauvegardé dans `BEST_MODEL_PATH`.
- **La dernière version** reste sauvegardée dans `MODEL_PATH`, pour reprendre l'entraînement.
- **Avec `RESUME = True`**, le meilleur score déjà obtenu est repris. Il est oublié si les réglages de l'évaluation ou l'objectif ont changé, car les scores ne seraient plus comparables.

L'évaluation coûte environ 20 secondes. Avec `EVAL_EVERY = 25`, cela représente un peu plus de 10 % du temps d'entraînement.

**Attention**, ces indicateurs dépendent aussi du nombre de prédateurs. Quand le réseau s'améliore, les prédateurs survivent et se reproduisent davantage, donc ils deviennent plus nombreux, et la nourriture se partage entre plus de monde. L'énergie moyenne peut donc baisser alors que le réseau progresse. La comparaison la plus juste est celle de l'évaluation (partie 9), où les deux politiques partent des mêmes mondes.

In [ ]:
def evaluate(policy, runs, steps, n_predators, seed=1000, record=False):
    # run `runs` simulations of the scenario where the predators follow `policy`, and measure the same
    # indicators as during the training; record=True also returns the populations of the first run
    totals = dict(predator_steps=0, alive_steps=0, energy=0.0, reward=0.0, reproductions=0, deaths=0,
                  steps=0, preys=0, predators=0)
    cooperation = dict(predator_steps=0, grouped=0, kills=0, group_kills=0, meals=0, shared_meals=0)
    survived, lengths, populations = 0, [], None
    for run in range(runs):
        np.random.seed(seed + run)
        world = new_world(n_predators)
        world.predator_policy = policy
        counts = []
        for t in range(steps):
            result = step_and_reward(world)
            out = result["out"]
            counts.append((out[0], out[1]))
            totals["predator_steps"] += len(result["alive"])
            totals["alive_steps"] += int(result["alive"].sum())
            totals["energy"] += result["energy"].sum()
            totals["reward"] += result["individual"].sum()
            totals["reproductions"] += int(result["reproduced"].sum())
            totals["deaths"] += int((~result["alive"]).sum())
            totals["steps"] += 1
            totals["preys"] += out[0]
            totals["predators"] += out[1]
            for key, event_count in cooperation_events(world).items():
                cooperation[key] += event_count
            if out[0] == 0 or out[1] == 0:
                break
        survived += (out[0] > 0 and out[1] > 0)
        lengths.append(t + 1)
        if record and run == 0:
            populations = np.array(counts)
    n = max(totals["predator_steps"], 1)
    result = dict(
        energy=float(totals["energy"] / max(totals["alive_steps"], 1)),
        individual_reward=float(totals["reward"] / n),
        reproductions_per_1000=float(1000 * totals["reproductions"] / n),
        deaths_per_1000=float(1000 * totals["deaths"] / n),
        survival=survived / runs,
        episode_length=float(np.mean(lengths)),
        predators=totals["predators"] / totals["steps"],
        preys=totals["preys"] / totals["steps"],
        **cooperation_indicators(cooperation),
    )
    return (result, populations) if record else result


start = time.time()
reference = evaluate(predator_heuristic, runs=N_WORLDS, steps=EPISODE_STEPS, n_predators=N_PREDATORS_INIT)
print(f"reference (hand-written rule), {time.time() - start:.0f} s:")
print({key: round(value, 3) for key, value in reference.items()})

In [ ]:
EVALUATION = dict(runs=EVAL_RUNS, steps=EVAL_STEPS, seed=EVAL_SEED, n_predators=N_PREDATORS_INIT, reward=REWARD)


def evaluation_score(brain):
    # reward per predator step of the network without exploration, always on the same worlds
    state = np.random.get_state()  # the evaluation sets the random seed: the training must not be affected
    score = evaluate(NeuralPredatorPolicy(brain, device=device), runs=EVAL_RUNS, steps=EVAL_STEPS,
                     n_predators=N_PREDATORS_INIT, seed=EVAL_SEED)["individual_reward"]
    np.random.set_state(state)
    brain.train()
    return score


if RESUME and os.path.exists(MODEL_PATH):
    brain, info = load_brain(MODEL_PATH, device)
    print("training resumed from", MODEL_PATH, info)
else:
    brain, info = PredatorBrain().to(device), {}
optimizer = torch.optim.Adam(brain.parameters(), lr=LEARNING_RATE)
training_worlds = [TrainingWorld() for _ in range(N_WORLDS)]

# best score so far (kept when the training is resumed, if it was measured the same way)
best_score = -np.inf
if RESUME and os.path.exists(BEST_MODEL_PATH):
    best_info = load_brain(BEST_MODEL_PATH, device)[1]
    if best_info.get("evaluation") == EVALUATION:
        best_score = best_info["score"]
        print(f"best network so far: iteration {best_info['iterations']}, score {best_score:.5f}")

# counters since the beginning of the training (also when it is resumed)
iteration = info.get("iterations", 0)
transitions = info.get("transitions", 0)
log = []
start = time.time()
while time.time() - start < TRAIN_MINUTES * 60:
    brain.train()
    batch, stats = collect_rollout(training_worlds, brain)
    compute_advantages(batch)
    losses = ppo_update(brain, optimizer, batch)
    iteration += 1
    transitions += stats["transitions"]
    log.append(dict(iteration=iteration, minutes=(time.time() - start) / 60, total_transitions=transitions,
                    **stats, **losses))

    # regular evaluation: keep the best network
    if iteration % EVAL_EVERY == 0:
        score = evaluation_score(brain)
        log[-1]["eval_score"] = score
        if score > best_score:
            best_score = score
            save_brain(BEST_MODEL_PATH, brain, reward=REWARD, iterations=iteration, transitions=transitions,
                       score=score, evaluation=EVALUATION)
        print(f"      evaluation at iteration {iteration}: score {score:.5f} (best {best_score:.5f})")

    if iteration % 10 == 1:
        e = log[-1]
        print(f"{e['minutes']:5.1f} min | it {iteration:4d} | {transitions / 1e6:6.2f} M transitions | "
              f"energy {e['energy']:5.1f} | repro/1000 {e['reproductions_per_1000']:5.2f} | "
              f"deaths/1000 {e['deaths_per_1000']:5.2f} | reward {e['individual_reward']:.4f} | "
              f"group kills {e['group_kills']:.2f} | speed {e['speed']:.2f} | episode {e['episode_length']:6.0f} | "
              f"kl {e['kl']:.4f} | std {e['std']:.2f}")
    if iteration % 20 == 0:
        save_brain(MODEL_PATH, brain, reward=REWARD, iterations=iteration, transitions=transitions)

save_brain(MODEL_PATH, brain, reward=REWARD, iterations=iteration, transitions=transitions)
print(f"\n{iteration} iterations, {transitions / 1e6:.1f} M transitions, saved in {MODEL_PATH}")
if os.path.exists(BEST_MODEL_PATH):
    print(f"best network: score {best_score:.5f}, in {BEST_MODEL_PATH}")

In [ ]:
# training curves (moving average over 10 iterations), with the level of the hand-written rule
def smooth(values, window=10):
    # moving average that ignores the missing values (nan: e.g. no episode ended during the iteration)
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    valid = ~np.isnan(values)
    total = np.convolve(np.where(valid, values, 0), np.ones(window), mode="valid")
    count = np.convolve(valid.astype(float), np.ones(window), mode="valid")
    return np.where(count > 0, total / np.maximum(count, 1), np.nan)


x = np.array([e["total_transitions"] for e in log]) / 1e6
curves = [
    ("energy", "Mean energy of the predators"),
    ("reproductions_per_1000", "Reproductions per 1000 predator steps"),
    ("deaths_per_1000", "Deaths per 1000 predator steps"),
    ("individual_reward", "Individual reward per step"),
    ("episode_length", "Episode length (time steps)"),
    ("predators_per_world", "Predators per world"),
    ("hunting_together", "Hunting together (kills nearby / time nearby)"),
    ("shared_meals", "Meals shared with another predator"),
    ("std", "Exploration (std of the actions)"),
    ("eval_score", "Evaluation score (without exploration)"),
]
fig, axes = plt.subplots(2, 5, figsize=(24, 8))
for ax, (key, title) in zip(axes.flat, curves):
    if key == "eval_score":
        # only measured every EVAL_EVERY iterations; the star is the best network
        scores = np.array([e.get("eval_score", np.nan) for e in log])
        measured = ~np.isnan(scores)
        ax.plot(x[measured], scores[measured], "o-", color="purple", label="network")
        if measured.any():
            best = np.nanargmax(scores)
            ax.plot(x[best], scores[best], "*", color="gold", markersize=18, markeredgecolor="black", label="best")
        ax.axhline(reference["individual_reward"], color="gray", ls="--", label="hand-written rule")
        ax.set_title(title)
        ax.set_xlabel("transitions (millions)")
        ax.legend(fontsize=8)
        continue
    y = smooth([e[key] for e in log])
    ax.plot(x[len(x) - len(y):], y, color="purple", label="network")
    if key == "hunting_together":
        ax.axhline(1, color="black", lw=0.5)
    if key in reference:
        ax.axhline(reference[key], color="gray", ls="--", label="hand-written rule")
    ax.set_title(title)
    ax.set_xlabel("transitions (millions)")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 9. L'évaluation face à la règle écrite à la main

On compare la dernière version du réseau et sa meilleure version (sans exploration : il prend toujours sa décision moyenne) à la règle écrite à la main, dans deux situations :
- **les conditions d'entraînement** : 20 prédateurs au départ, épisodes de 2000 pas ;
- **le scénario de `Lotka-Volterra.ipynb`** : 7 prédateurs au départ, 5000 pas. On regarde aussi si l'écosystème survit et s'il oscille toujours.

Attention : un réseau qui chasse très bien peut **exterminer les proies**, puis mourir de faim. Bien jouer individuellement n'est pas forcément bon pour l'écosystème.

Les colonnes `predators` et `preys` donnent les populations moyennes. `survival` compte les simulations où les deux espèces existent encore à la fin, même s'il ne reste qu'un seul prédateur.

**Les indicateurs de coopération** :
- **`group kills`** : la part des proies tuées par un prédateur qui avait un autre prédateur à moins de `COOPERATION_RADIUS` (5).
- **`grouped`** : la part du temps où un prédateur a un autre prédateur à moins de 5, **à n'importe quel moment**. Des prédateurs qui vivent en groupe, par exemple en famille (un petit naît entre ses deux parents), ont un `grouped` élevé, sans coopérer pour autant.
- **`together`**, l'**indice de chasse groupée** : `group kills` / `grouped`. Il vaut 1 si les prédateurs ne sont pas plus proches les uns des autres quand ils tuent qu'en temps normal. Au-dessus de 1, ils se rapprochent **pour tuer**.
- **`shared`** : la part des repas pris sur une carcasse en même temps qu'un autre prédateur.

Même sans aucune coopération, plusieurs prédateurs qui poursuivent les mêmes groupes de proies se retrouvent proches au moment de tuer. La règle écrite à la main, qui ne coopère pas, donne donc le niveau de référence de `together`. Un réseau ne **chasse ensemble** que s'il le dépasse nettement.

In [ ]:
policies = [("hand-written rule", predator_heuristic),
            ("network (last)", NeuralPredatorPolicy(brain, device=device, deterministic=True))]
if os.path.exists(BEST_MODEL_PATH):
    best_brain, best_info = load_brain(BEST_MODEL_PATH, device)
    print(f"best network: iteration {best_info['iterations']}, score {best_info['score']:.5f}\n")
    policies.append(("network (best)", NeuralPredatorPolicy(best_brain, device=device, deterministic=True)))

rows = []
for name, policy in policies:
    rows.append((name, "training conditions", evaluate(policy, runs=4, steps=EPISODE_STEPS, n_predators=N_PREDATORS_INIT)))
    rows.append((name, "scenario (7 predators)", evaluate(policy, runs=3, steps=5000, n_predators=7)))

columns = {"energy": "energy", "reproductions_per_1000": "repro/1000", "deaths_per_1000": "deaths/1000",
           "individual_reward": "reward", "survival": "survival", "predators": "predators", "preys": "preys",
           "group_kills": "group kills", "grouped": "grouped", "hunting_together": "together", "shared_meals": "shared"}
print(f"{'':18s} {'conditions':24s}" + "".join(f"{label:>12s}" for label in columns.values()))
for name, conditions, result in rows:
    print(f"{name:18s} {conditions:24s}" + "".join(f"{result[key]:12.3f}" for key in columns))

In [ ]:
# populations in the scenario of Lotka-Volterra.ipynb, with each policy (the same two seeds for all)
fig, axes = plt.subplots(2, len(policies), figsize=(8 * len(policies), 8), sharey=True)
for row, seed in enumerate((1000, 1001)):
    for ax, (name, policy) in zip(axes[row], policies):
        _, populations = evaluate(policy, runs=1, steps=5000, n_predators=7, seed=seed, record=True)
        ax.plot(populations[:, 0], color="blue")
        ax2 = ax.twinx()
        ax2.plot(populations[:, 1], color="orange")
        ax.set_title(f"Predators controlled by the {name} (seed {seed})")
        ax.set_xlabel("Time")
        ax.set_ylabel("Preys", color="blue")
        ax2.set_ylabel("Predators", color="orange")
fig.tight_layout()
plt.show()

## 10. La vidéo des prédateurs entraînés

On simule le scénario avec les prédateurs contrôlés par le réseau (sa meilleure version, s'il y en a une), puis on en fait une vidéo avec `render_video`, comme dans `Lotka-Volterra.ipynb`.

In [ ]:
from render import render_video
from IPython.display import Video

np.random.seed(2000)
world = new_world(n_predators=7)
world.predator_policy = policies[-1][1]  # the best network if there is one, else the last version
history, frames = {}, {}
for t in range(3000):
    history[t] = world.step()
    frames[t] = world.return_dict()
    if history[t][0] == 0 or history[t][1] == 0:
        break

video_path = render_video(frames, history, world_width=WORLD_SIZE, world_height=WORLD_SIZE,
                          max_energy_predator=world.max_energy_predator,
                          output_path=f"./plot/trained_predators_{REWARD}.mp4", fps=30, every=5)
Video(video_path, embed=True, width=960)

## 11. Comparer tous les réseaux sauvegardés

Cette partie compare la règle écrite à la main et **tous les réseaux du dossier `models/`** (par exemple un réseau entraîné avec chaque objectif), sur les mêmes mondes : le scénario de `Lotka-Volterra.ipynb`, avec 6 graines de 5000 pas.

La récompense n'apparaît pas : elle n'a pas le même sens d'un objectif à l'autre. On compare donc ce qui se passe dans l'écosystème. Les réseaux sauvegardés avec d'anciennes entrées, qui ne peuvent plus être chargés, sont ignorés.

In [ ]:
candidates = [("hand-written rule", predator_heuristic)]
for path in sorted(glob.glob("./models/*.pt")):
    try:
        saved_brain, saved_info = load_brain(path, device)
    except RuntimeError:  # network saved with other inputs
        print("ignored (other inputs):", path)
        continue
    candidates.append((os.path.basename(path), NeuralPredatorPolicy(saved_brain, device=device)))

comparison = [(name, evaluate(policy, runs=6, steps=5000, n_predators=7, seed=3000)) for name, policy in candidates]

columns = {"energy": "energy", "reproductions_per_1000": "repro/1000", "deaths_per_1000": "deaths/1000",
           "survival": "survival", "predators": "predators", "preys": "preys",
           "group_kills": "group kills", "grouped": "grouped", "hunting_together": "together", "shared_meals": "shared"}
print(f"{'':34s}" + "".join(f"{label:>12s}" for label in columns.values()))
for name, result in comparison:
    print(f"{name:34s}" + "".join(f"{result[key]:12.3f}" for key in columns))

# populations with each policy, on the same world
fig, axes = plt.subplots(1, len(candidates), figsize=(6 * len(candidates), 4), squeeze=False)
for ax, (name, policy) in zip(axes[0], candidates):
    _, populations = evaluate(policy, runs=1, steps=5000, n_predators=7, seed=3000, record=True)
    ax.plot(populations[:, 0], color="blue")
    ax2 = ax.twinx()
    ax2.plot(populations[:, 1], color="orange")
    ax.set_title(name)
    ax.set_xlabel("Time")
    ax.set_ylabel("Preys", color="blue")
    ax2.set_ylabel("Predators", color="orange")
fig.tight_layout()
plt.show()

## Utiliser le réseau entraîné ailleurs

Dans `Lotka-Volterra.ipynb` (ou n'importe quel script), il suffit de remplacer la politique des prédateurs :

```python
from predator_brain import load_brain, NeuralPredatorPolicy

brain, info = load_brain("./models/predator_brain_energy_level_best.pt")
world.predator_policy = NeuralPredatorPolicy(brain)
```

## Pour aller plus loin

- **Entraîner plus longtemps** : mettre `RESUME = True` et augmenter `TRAIN_MINUTES`, puis relancer à partir de la boucle d'entraînement. Si le comportement change trop à chaque mise à jour (`kl` nettement au-dessus de 0,01), baisser `LEARNING_RATE` (par exemple à 1e-4).
- **Plus ou moins d'exploration** : changer `ENTROPY_COEF`. Si l'écart-type des actions (`std`) s'effondre, l'augmenter ; si le réseau reste trop hésitant, le baisser.
- **Comparer les objectifs** : entraîner un réseau avec chaque valeur de `REWARD`, puis lancer la partie 11.
- **Encourager plus la coopération** : augmenter `GROUP_SHARE` (la part collective de la récompense).
- **Rendre la reproduction plus ou moins attractive** : changer `REPRODUCTION_BONUS`.
- **Voir plus de voisins** : augmenter `K_PREYS`, `K_PREDATORS` et `K_CARCASSES` dans `predator_brain.py`. Il faut alors repartir de zéro, car le réseau change de taille.